In [57]:
import networkx as nx
import itertools

from bp.world import random_grid_world, Scenario

demands = {
    ((1, 0), (1, 3)): 6,
    ((0, 0), (2, 3)): 6,
}

world = random_grid_world(
    rows=4,
    cols=4,
    demands=demands,
    seed=0,
)
G = world.network.graph
world.network.bpr_beta = 1
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
for od, n_k in demands.items():
    print(f"Demand for {od}: {n_k}")

world.total_population=12
Demand for ((1, 0), (1, 3)): 6
Demand for ((0, 0), (2, 3)): 6


In [58]:
accident_congestion = 10
target_edge = ((1, 1), (1, 2))

travel_time = dict(nominal.travel_time)
travel_time[target_edge] += accident_congestion

accident = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal.discomfort,
    hazard=nominal.hazard,
    cost=nominal.cost,
    emissions=nominal.emissions,
    policing=nominal.policing
)

scenarios = {
    "nominal": (nominal, .8),
    "accident": (accident, .2),
}

assert all(prior >= 0 for _, prior in scenarios.values()), "invalid prior distribution"
assert sum(prior for _, prior in scenarios.values()) == 1, "invalid prior distribution"

In [59]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("asymmetric dictator (anonymous)")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

In [60]:
from collections.abc import Mapping, Sequence

from bp.world import Arc, Node

def edge_path(path: Sequence[Node]) -> Sequence[Arc]:
    return list(itertools.pairwise(path))

# select k shortest paths as possible routes
K = 5
paths_per_od = {}
for origin, dest in demands.keys():
    paths = [edge_path(p) for p in itertools.islice(nx.shortest_simple_paths(G, origin, dest, weight="travel_time"), K)]
    paths_per_od[(origin, dest)] = paths
# TODO: shortest paths are calculated based on the nominal state, which may not be true under the accident state

active_arcs = set(
    itertools.chain.from_iterable(
        itertools.chain.from_iterable(paths_per_od.values())
    )
)

profiles_per_od = {
    od: list(itertools.combinations_with_replacement(paths_per_od[od], n_k))
    for od, n_k in demands.items()
}

joint_action_profiles = list(itertools.product(*profiles_per_od.values()))

action_profile_flows: Sequence[Mapping[Arc, int]] = []
for joint_profile in joint_action_profiles:
    flow = {a: 0 for a in active_arcs}
    for od_profile in joint_profile:
        for path in od_profile:
            for a in path:
                flow[a] += 1
    action_profile_flows.append(flow)

for od, paths in paths_per_od.items():
    print(od)
    for i, path in enumerate(paths):
        print("\t", f"{i}:", path)
    print("\n")

print(f"{len(active_arcs)=} ({len(active_arcs) / (len(A) / 2) * 100:.2f}% of arcs)") # / 2 b/c includes backwards edges which are probably irrelevant
print(f"{len(profiles_per_od)=}")
print(f"{len(joint_action_profiles)=}")

((1, 0), (1, 3))
	 0: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
	 1: [((1, 0), (0, 0)), ((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3))]
	 2: [((1, 0), (1, 1)), ((1, 1), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3))]
	 3: [((1, 0), (1, 1)), ((1, 1), (2, 1)), ((2, 1), (2, 2)), ((2, 2), (1, 2)), ((1, 2), (1, 3))]
	 4: [((1, 0), (1, 1)), ((1, 1), (2, 1)), ((2, 1), (2, 2)), ((2, 2), (2, 3)), ((2, 3), (1, 3))]


((0, 0), (2, 3))
	 0: [((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3)), ((1, 3), (2, 3))]
	 1: [((0, 0), (1, 0)), ((1, 0), (1, 1)), ((1, 1), (2, 1)), ((2, 1), (2, 2)), ((2, 2), (2, 3))]
	 2: [((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (1, 2)), ((1, 2), (2, 2)), ((2, 2), (2, 3))]
	 3: [((0, 0), (1, 0)), ((1, 0), (2, 0)), ((2, 0), (2, 1)), ((2, 1), (2, 2)), ((2, 2), (2, 3))]
	 4: [((0, 0), (1, 0)), ((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (2, 2)), ((2, 2), (2, 3))]


len(active_arcs)=20 (83.33% of arcs)
l

In [61]:
# reminder: beta=1 is fixed
# tau[omega, a, k] := average cost for k players on arc a under state omega
tau = {}
for scenario_name, (omega, _) in scenarios.items():
    for a in active_arcs:
        for k in range(n + 2): # n + 1 + 1 b/c of deviation indicator variable
            tau[scenario_name, a, k] = omega.travel_time[a] * (1 + alpha * ((k - 1) / c[a]) ** beta)

In [ ]:
# decision: phi[omega, a] := \prob[a \mid \theta]
phi = {}
for scenario_name in scenarios:
    for action_profile_idx in range(len(joint_action_profiles)):
        phi[scenario_name, action_profile_idx] = model.addVar(
            vtype=GRB.CONTINUOUS, lb=0, ub=1,
            name=f"phi_{scenario_name}_{action_profile_idx}"
        )

# constraint: \sum_{a \in A} phi[\omega, a] = 1
for scenario_name in scenarios:
    model.addConstr(
        gp.quicksum(
            phi[scenario_name, action_profile_idx]
            for action_profile_idx in range(len(joint_action_profiles))
        ) == 1,
        name=f"phi_{scenario_name}"
    )

# objective: expected social cost
expected_social_cost = gp.LinExpr()
for scenario_name, (omega, mu) in scenarios.items():
    for action_profile_idx, action_profile_flow in enumerate(action_profile_flows):
        cost = sum( # c_{\omega}(a)
            flow * tau[scenario_name, a, flow]
            for a, flow in action_profile_flow.items()
        )
        expected_social_cost += mu * phi[scenario_name, action_profile_idx] * cost

model.setObjective(expected_social_cost, GRB.MINIMIZE)

In [63]:
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

In [65]:
print(f"{model.ObjVal=:.2f}")
print(f"{model.Runtime=:.3f}s")

for scenario_name in scenarios:
    print(f"\nState: {scenario_name}")

    for action_profile_idx, joint_profile in enumerate(joint_action_profiles):
        prob = phi[scenario_name, action_profile_idx].X
        if prob > 1e-3:
            print(f"\tJoint Profile {action_profile_idx}: ({prob=:.3f})")
            for od, od_profile in zip(profiles_per_od.keys(), joint_profile):
                path_flows = tuple(od_profile.count(path) for path in paths_per_od[od])
                print("\t\t", f"{od}:", path_flows)

model.ObjVal=350.21
model.Runtime=0.036s

State: nominal
	Joint Profile 48: (prob=1.000)
		 ((1, 0), (1, 3)): (6, 0, 0, 0, 0)
		 ((0, 0), (2, 3)): (2, 1, 1, 2, 0)

State: accident
	Joint Profile 19613: (prob=1.000)
		 ((1, 0), (1, 3)): (1, 1, 2, 2, 0)
		 ((0, 0), (2, 3)): (1, 2, 1, 2, 0)
